In [ ]:
from openai import OpenAI

client = OpenAI(api_key="")

print("Client", client)

Client <openai.OpenAI object at 0x000001EF72898590>


In [3]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful ai assistant."},
        {"role": "user", "content": "Hi, my name is chetan"},
        {"role": "assistant", "content": "Hello Chetan! How can I assist you today?"},
        {"role": "user", "content": "What is my name?"}
    ]
)

print(response.choices[0].message.content)

Your name is Chetan. How can I assist you today, Chetan?


In [10]:
print(response.model)
print(response.choices[0].finish_reason)
print(response.usage)

gpt-4o-mini-2024-07-18
stop
CompletionUsage(completion_tokens=11, prompt_tokens=35, total_tokens=46, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))


In [25]:
for temp in [0, 1, 1.5]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful ai assistant. Give one line response only"},
            {"role": "user", "content": "What is python"}
        ],
        temperature=temp
    )
    print(f"Temp: {temp} = ", response.choices[0].message.content)

Temp: 0 =  Python is a high-level, interpreted programming language known for its readability and versatility.
Temp: 1 =  Python is a high-level, interpreted programming language known for its readability and versatility.
Temp: 1.5 =  Python is a high-level, interpreted programming language known for its readability and versatility.


In [35]:
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful ai assistant."},
        {"role": "user", "content": "Hi my name is chetan"}
    ],
    stream=True
)

full_response = ""

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        full_response += delta
        print(full_response)


# print("\n\n=================full response===================")
# print(full_response)


Hello
Hello C
Hello Chet
Hello Chetan
Hello Chetan!
Hello Chetan! How
Hello Chetan! How can
Hello Chetan! How can I
Hello Chetan! How can I assist
Hello Chetan! How can I assist you
Hello Chetan! How can I assist you today
Hello Chetan! How can I assist you today?


In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful ai assistant."},
        {"role": "user", "content": "Tell me joke, baseline then puchlne"}
    ]
)

print(response.choices[0].message.content)

Sure! Here’s a classic joke:

**Baseline:** Why did the scarecrow win an award?

**Punchline:** Because he was outstanding in his field!


In [48]:
from pydantic import BaseModel, Field

class Hotel(BaseModel):
    adults: int = Field(description="The number of adults in the hotel")
    children: int = Field(description="The number of children in the hotel")
    checkin: str = Field(description="The checkin date of the hotel")
    checkout: str = Field(description="The checkout date of the hotel")

print("Schema", Hotel.model_json_schema())

Schema {'properties': {'adults': {'description': 'The number of adults in the hotel', 'title': 'Adults', 'type': 'integer'}, 'children': {'description': 'The number of children in the hotel', 'title': 'Children', 'type': 'integer'}, 'checkin': {'description': 'The checkin date of the hotel', 'title': 'Checkin', 'type': 'string'}, 'checkout': {'description': 'The checkout date of the hotel', 'title': 'Checkout', 'type': 'string'}}, 'required': ['adults', 'children', 'checkin', 'checkout'], 'title': 'Hotel', 'type': 'object'}


In [ ]:
response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful ai assistant."},
        {"role": "user", "content": "I want to book a hotel for 2 adults and 1 child from 1st to 3rd of September 2026"}
    ],
    response_format=Hotel
)

hotel = response.choices[0].message.parsed

print("adults: ", hotel.adults)
print("children: ", hotel.children)
print("checkin: ", hotel.checkin)
print("checkout: ", hotel.checkout)



prompt = "My name is chetan"
response = "Person name is chatan" "Chetan"
response = "{function_name: 'get_latest_news', question: 'What is the latest news?'}"






In [15]:
def get_latest_news(query):
    # we can call some news api to fetch the latest news
    return "This is testing of latest news feature, response according to the query"

def web_search(query):
    # we can call some web search api to fetch the results
    return "results"


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_latest_news",
            "description": "Call this function when user asks for latest news",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Query to get latest news"
                    },
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Call this function when user asks for web search",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Query to get web search results"
                    },
                },
                "required": ["query"]
            }
        }
    },
]

response =client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful ai assistant."
        },
        {
            "role": "user",
            "content": "fetch latest news for me"
        }
    ],
    tools=tools,
    tool_choice="auto"
)

print(response.choices[0].message.content)

if response.choices[0].message.tool_calls:
    funtion_name = response.choices[0].message.tool_calls[0].function.name
    query = response.choices[0].message.tool_calls[0].function.arguments

    tool_response = None

    if funtion_name == "get_latest_news":
        tool_response = get_latest_news(query)

    elif funtion_name == "web_search":
        tool_response = web_search(query)

    print(tool_response)



    response =client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful ai assistant."
            },
            {
                "role": "user",
                "content": "fetch latest news for me"
            },
            {
                "role": "assistant",
                "content": f"This is tool response {tool_response}, answer user according to the tool response"
            }
        ]
    )

    print(response.choices[0].message.content)






None
This is testing of latest news feature, response according to the query
I'm unable to browse the internet or fetch real-time news updates. However, I recommend checking reliable news websites or apps like BBC, CNN, Reuters, or your favorite news outlet for the latest updates. If you're looking for news on a specific topic, feel free to ask, and I can provide some general information based on knowledge up to October 2023.
